### Tools
LLMs can ask to use tools for tasks like getting data from APIs, searching online, querying databases, or executing code. A tool usually consists of two parts:

1. A definition (schema) that explains the tool — including its name, purpose, and expected inputs
2. The actual function or async function that runs when the tool is called

This helps the model interact with external systems and perform real-world actions. ([LangChain Docs][1])

[1]: https://docs.langchain.com/oss/python/langchain/tools?utm_source=chatgpt.com "Tools - Docs by LangChain"


In [3]:
from langchain.chat_models import init_chat_model
from dotenv import load_dotenv
load_dotenv()

# api_key = os.getenv("GROQ_API_KEY")

model = init_chat_model("groq:qwen/qwen3-32b")
response = model.invoke("What is apple?")
response

AIMessage(content='<think>\nOkay, the user is asking "What is apple?" Let me start by considering the different possible interpretations. The most straightforward is Apple Inc., the tech company. However, there\'s also the fruit, and maybe even other contexts like the Apple TV or Apple products.\n\nFirst, I should confirm if they\'re referring to the company or the fruit. Since the question is in English and using the lowercase "apple," it might be the fruit, but Apple Inc. is also a common topic. Maybe the user is a non-native speaker or there\'s a typo. Let me check the context. The user just asked "What is apple?" without any prior conversation, so I need to cover both possibilities.\n\nFor Apple Inc., I can mention it\'s a multinational tech company founded in 1976 by Steve Jobs, Steve Wozniak, Ronald Wayne. They\'re known for products like iPhone, iPad, Mac computers, and services like the App Store and Apple Music. Also, their headquarters in Cupertino, California.\n\nFor the fru

In [4]:
from langchain.tools import tool

@tool
def get_population(location:str)->str:
    """Get the current population in a given location"""
    populations = {
        "delhi": "33 million",
        "mumbai": "21 million",
        "new york": "8.5 million"
    }
    return populations.get(
        location.lower(),
        f"Population data for {location} not found."
    )


model_with_tools=model.bind_tools([get_population])

c:\Users\Administrator\Downloads\b150-genaiops\18_Langchain\.venv\Lib\site-packages\langgraph\checkpoint\serde\encrypted.py:5: LangChainPendingDeprecationWarning: The default value of `allowed_objects` will change in a future version. Pass an explicit value (e.g., allowed_objects='messages' or allowed_objects='core') to suppress this warning.
  from langgraph.checkpoint.serde.jsonplus import JsonPlusSerializer


In [6]:
response = model_with_tools.invoke("What's the population in new york ?")
print(response)
for tool_call in response.tool_calls:
    # View tool calls made by the model
    print(f"Tool: {tool_call['name']}")
    print(f"Args: {tool_call['args']}")

content='' additional_kwargs={'reasoning_content': 'Okay, the user is asking for the population in New York. Let me check the tools available. There\'s a function called get_population that takes a location parameter. The required parameter is location, and it\'s a string. So I need to call get_population with "New York" as the location. I should make sure the arguments are correctly formatted as JSON. Let me structure the tool call accordingly.\n', 'tool_calls': [{'id': 'pa72870d2', 'function': {'arguments': '{"location":"New York"}', 'name': 'get_population'}, 'type': 'function'}]} response_metadata={'token_usage': {'completion_tokens': 106, 'prompt_tokens': 156, 'total_tokens': 262, 'completion_time': 0.149528351, 'completion_tokens_details': {'reasoning_tokens': 81}, 'prompt_time': 0.006984261, 'prompt_tokens_details': None, 'queue_time': 0.075739392, 'total_time': 0.156512612}, 'model_name': 'qwen/qwen3-32b', 'system_fingerprint': 'fp_d58dbe76cd', 'service_tier': 'on_demand', 'fin

## Simple Restaurant Analogy 🍽️

Think of the AI model as a **customer in a restaurant**.

---

## Step 1: Customer places order

```python
ai_msg = model_with_tools.invoke(messages)
```

The customer says:

> “I want to know the population of New York.”

But the customer does not know the answer directly.

So the waiter (AI model) decides:

> “I should ask the population department/tool.”

This is the tool call generation step.

---

## Step 2: Restaurant staff does the actual work

```python
tool_result = get_population.invoke(tool_call)
```

Now the kitchen/tool executes the task:

> “Population of New York is 8.5 million.”

The tool fetches the real information.

---

## Step 3: Waiter returns final answer

```python
final_response = model_with_tools.invoke(messages)
```

The waiter takes the tool result and responds nicely to the customer:

> “The current population of New York is around 8.5 million.”

This is the final AI response generation step.

---

# Another Simple Office Analogy 🏢

| Component      | Real-world analogy                  |
| -------------- | ----------------------------------- |
| User           | Customer                            |
| LLM            | Smart receptionist                  |
| Tool           | Specialized employee                |
| Tool execution | Employee doing actual work          |
| Final response | Receptionist giving polished answer |

---

# Flow in One Line

```text
User asks → AI decides tool needed → Tool executes → AI formats final response
```

This is the core idea behind tool calling in LangChain and AI agents. ([langchain.com][1])

[1]: https://www.langchain.com/blog/tool-calling-with-langchain?utm_source=chatgpt.com "Tool Calling with LangChain"


### Tool Execution Loops

In [ ]:
# Step 1: Model generates tool calls
messages = [{"role": "user", "content": "What's the population in new york ?"}]
ai_msg = model_with_tools.invoke(messages)
messages.append(ai_msg)

# Step 2: Execute tools and collect results
for tool_call in ai_msg.tool_calls:
    # Execute the tool with the generated arguments
    tool_result = get_population.invoke(tool_call)
    messages.append(tool_result)

# Step 3: Pass results back to model for final response
final_response = model_with_tools.invoke(messages)
print(final_response.text)

The population of New York is approximately 8.5 million.


In [ ]:
messages

[{'role': 'user', 'content': "What's the weather in Boston?"},
 AIMessage(content='', additional_kwargs={'reasoning_content': 'Okay, the user is asking for the weather in Boston. I need to use the get_weather function. Let me check the function parameters. The required parameter is location, which should be a string. Boston is the location here. So I\'ll call the function with location set to "Boston". Make sure the JSON is correctly formatted with the function name and arguments. No other functions are available, so this should be straightforward.\n', 'tool_calls': [{'id': '20cfda4p0', 'function': {'arguments': '{"location":"Boston"}', 'name': 'get_weather'}, 'type': 'function'}]}, response_metadata={'token_usage': {'completion_tokens': 109, 'prompt_tokens': 153, 'total_tokens': 262, 'completion_time': 0.199966855, 'completion_tokens_details': {'reasoning_tokens': 85}, 'prompt_time': 0.006519155, 'prompt_tokens_details': None, 'queue_time': 0.055720395, 'total_time': 0.20648601}, 'mod